In [227]:
import pandas as pd
import numpy as np
import datetime
import numpy as np
# import matplotlib.pyplot as plt
# import scipy.stats as stats

In [228]:
assets = pd.read_parquet("melted_all_files_assets.parquet")
income = pd.read_parquet("melted_all_files_ie.parquet")
liab = pd.read_parquet("melted_all_files_liabilities.parquet")

In [229]:
df_combined = pd.concat([assets, liab, income], ignore_index=True)

In [230]:
df_combined = df_combined[df_combined['Bank_name'] != 'Всего по БВУ:']

In [231]:
df_combined['Measure'].unique()

<ArrowStringArray>
[                                                                     'Активы',
                                                      'Активы БВУ в ин.валюте',
                                                  'Активы до года, тыс. тенге',
                                       'Активы от 1 года до 3 лет, тыс. тенге',
                                              'Активы свыше 3 лет, тыс. тенге',
                             'Активы в иностранной валюте до года, тыс. тенге',
                  'Активы в иностранной валюте от 1 года до 3 лет, тыс. тенге',
                         'Активы в иностранной валюте свыше 3 лет, тыс. тенге',
                  'Активы со сроком до погашения до востребования, тыс .тенге',
                       'Активы со сроком до погашения до 1 месяца, тыс .тенге',
 ...
                        'Непредвиденные статьи (на конец периода, тыс. тенге)',
    'Чистый доход до уплаты подоходного налога (на конец периода, тыс. тенге)',
        'Расходы

In [232]:
df_combined['Sheet'].unique()

<ArrowStringArray>
[                         'Активы',          'Активы БВУ в ин.валюте',
       'Активы до 1, 1-3, свыше 3',    'Активы в ин.валюте по срокам',
                'Активы по срокам',    'Требования к НБ РК по срокам',
   'Требования к банкам по срокам',                    'ЦБ по срокам',
          'Инв в К и СД по срокам',         'Обратное РЕПО по срокам',
 ...
 'Расходы по обязательствам перед', 'Расходы в виде вознаграждения п',
 'Расходы по ценным бумагам и суб', 'Расходы по по операциям "РЕПО"(',
 'Операционные расходы (на конец ', 'Чистый % доход, связанный с пол',
 'Чистый непроцентный доход (на к', 'Доходы от восстановления провиз',
 'Расходы на формирование провизи', 'Чистый доход�убыток по формиров']
Length: 184, dtype: str

In [233]:
# REMOVING DUPLICATES AND SOME TRASH
df_combined = df_combined[~((df_combined['Sheet'] == 'Займы с просрочкой') & (df_combined['Measure'] == 'Итого ссудный портфель'))]
df_combined = df_combined[~((df_combined['Sheet'] == 'Прочая пром-ть') & (df_combined['Measure'] == 'Итого ссудный портфель'))]

In [234]:
df_combined = df_combined[
    ~(
        df_combined['Sheet'].isin([
            'Домохозяйства', 'Горнодобыв.пром-ть', 'Займы физ.лицам',
            'Займы физ.лицам на строит-во', 'Ипотечные займы', 'Непроизводственная сфера',
            'Обраб.пром-ть', 'Операции с недвиж.имущ.', 'Потреб.займы на прочие цели',
            'Потреб.цели', 'Потреб.цели_без обеспечения', 'Промышленность',
            'Прочая пром-ть', 'Свод_СП_6 отр.', 'Сельское хозяйство',
            'Строительство', 'Торговля', 'Транспорт и Связь'
        ])
        &
        (df_combined['Measure'] == 'Сумма займов, по которым просроченная задолженность свыше 90 дней')
    )
]

In [235]:
df_combined = df_combined[
    ~(
        df_combined['Sheet'].isin([
            'NPL_ЮЛ_ФЛ_МСБ_1'
        ])
        &
        (df_combined['Measure'] == 'Итого NPL по ссудному портфелю')
         &
        (df_combined['SourceFile'] == 'Assets_MID.xlsx')
    )
]

In [236]:
# NORMALAZING MEAUSURES==

In [237]:
normalize_fields = {
    # Прочие финансовые активы
    "Прочие финансовые активы от 1 года до 3 лет": "Прочие финансовые активы от 1 года до 3 лет, тыс. тенге",
    "Прочие финансовые активы от 1 до 30 дней, тыс. тенг": "Прочие финансовые активы от 1 до 30 дней, тыс. тенге",
    "Прочие финансовые активы от 31 до 90 дней, тыс. тенге": "Прочие финансовые активы от 31 до 90 дней, тыс. тенге",
    "Прочие финансовые активы от 3 месяцев до 1 года, тыс. тенге": "Прочие финансовые активы от 3 месяцев до 1 года, тыс. тенге",
    "Прочие финансовые активы свыше 3 лет, тыс. тенге": "Прочие финансовые активы свыше 3 лет, тыс. тенге",
    "Прочие финансовые активы, по которым есть просрочка платежей свыше 30 дней, тыс. тенге": "Прочие финансовые активы, по которым есть просрочка платежей свыше 30 дней, тыс. тенге",

    # Займы
    "Займы, предоставленнные юридическим и физическим лицам (за вычетом резервов (провизий))": "Займы, предоставленные юридическим и физическим лицам (за вычетом резервов (провизий)), тыс. тенге",
    "Займы с просроченной задолженностью свыыше 180 дней (для сведения)": "Займы с просроченной задолженностью свыше 180 дней (для сведения)",
    "Расходы по обязательствам перед клиентамия (на конец периода, тыс. тенге)": "Расходы по обязательствам перед клиентами (на конец периода, тыс. тенге)",

    # Скобки и формат
    "Требования к Национальному Банку Республики Казахстан) от 1 до 30 дней, тыс. тенге": "Требования к Национальному Банку Республики Казахстан от 1 до 30 дней, тыс. тенге",
    "Инвестиции в капитал и Субординированный долг (за вычетом резервов (провизий), тыс. тенге": "Инвестиции в капитал и Субординированный долг (за вычетом резервов (провизий)), тыс. тенге",
    "Операции «обратное РЕПО» (за вычетом резервов (провизий) свыше 3 лет, тыс. тенге": "Операции «обратное РЕПО» (за вычетом резервов (провизий)) свыше 3 лет, тыс. тенге",

    # NPL и провизии
    "NPL_90_Prov_296": "NPL_90_Prov_Пр.№296",
    "NPL 180+_для сведения": "NPL 180+_для сведения",
    "NPL 180+_Prov_IFRS_для сведения": "NPL 180+_Prov_IFRS_для сведения",
    "Итого Провизии по займам с просроченной задлолженностью свыше 90 дней": "Итого Провизии по займам с просроченной задолженностью свыше 90 дней",

    # Дублирующиеся категории
    "Банковские займы субъектам МСБ": "Банковские займы субъектам малого предпринимательства",
    "Вклады физ.лиц и юр.лиц в ин.валюте, тыс. тенге": "Вклады всего в ин.валюте, тыс.тенге",
    'Вклады физ.лиц и юр.лиц в ин.валюте, тыс.тенге': "Вклады всего в ин.валюте, тыс.тенге",
                                                    

    # Доходы и расходы
    "Доходы, связанные с получением вознаграждения": "Доходы, связанные с получением вознаграждения (на конец периода, тыс. тенге)",
    "Расходы, связанные с выплатой вознаграждения": "Расходы, связанные с выплатой вознаграждения (на конец периода, тыс. тенге)",
    "Расходы, не связанные с выплатой вознаграждения": "Расходы, не связанные с выплатой вознаграждения (на конец периода, тыс. тенге)",
    "Процентные доходы по займам": "Процентные доходы по займам (на конец периода, тыс. тенге)",
    
    "Чистый % доход, связанный с получением вознаграждения": "Чистый % доход, связанный с получением вознаграждения",
    'Чистый процентный доход (на конец периода, тыс. тенге)':'Чистый % доход, связанный с получением вознаграждения',
    
    "Процентные расходы по сберегательным вкладам": "Процентные расходы по сберегательным вкладам (на конец периода, тыс. тенге)",

    'Обязательства, связанные с выплатой % вознаграждения ':'Обяз-ва, связ. с выпл.%возн.',
    'Обязательства, связанные с выплатой % вознаграждения':'Обяз-ва, связ. с выпл.%возн.',

    'Итого NPL по ссудному портфелю': 'Сумма займов, по которым просроченная задолженность свыше 90 дней',

    'Требования к НБ':'Требования к НБ (на конец периода, тыс. тенге)',
    'Корр. Счета и вклады в БВУ и НБО (на конец периода, тыс. тенге)': 'Корр. Счета и вклады, размещенные в других БВУ и НБО (на конец периода, тыс. тенге)'
}


In [238]:
df_combined['Measures_normalized'] = df_combined['Measure'].map(lambda x: normalize_fields.get(x, x))

In [239]:
# creating mapping of similar bank names =========================================

In [ ]:
mapping = {


}

In [241]:
df_combined['BVU'] = df_combined['Bank_name'].replace(mapping)

In [242]:
# removing duplicate rows ----------------------------------------------------------------

In [243]:
len(df_combined)

1781128

In [244]:
df_combined = df_combined.drop_duplicates(subset=['Date','Measures_normalized','Amount','BVU'], keep='last')

In [245]:
len(df_combined)

1563705

In [ ]:
df_filtered = df_combined[df_combined['Date'] >= '2006-01-01']

In [247]:
# making format date -------------------------------------------------

In [248]:
df_filtered.Date.unique()

<DatetimeArray>
['2008-01-01 00:00:00', '2008-01-31 00:00:00', '2008-02-29 00:00:00',
 '2008-03-31 00:00:00', '2008-04-30 00:00:00', '2008-05-31 00:00:00',
 '2008-06-30 00:00:00', '2008-07-31 00:00:00', '2008-08-31 00:00:00',
 '2008-09-30 00:00:00',
 ...
 '2025-04-01 00:00:00', '2025-05-01 00:00:00', '2025-06-01 00:00:00',
 '2025-07-01 00:00:00', '2025-08-01 00:00:00', '2025-09-01 00:00:00',
 '2025-10-01 00:00:00', '2025-11-01 00:00:00', '2025-12-01 00:00:00',
 '2026-02-01 00:00:00']
Length: 463, dtype: datetime64[ns]

In [249]:
df_filtered['Date'] = pd.to_datetime(df_filtered['Date'])

df_filtered['End_month'] = df_filtered['Date'].where(
    df_filtered['Date'].dt.day != 1,
    df_filtered['Date'] - pd.offsets.MonthEnd(1)
)

In [250]:
# MAPPING OF DUPLICATES WE FOUND LATER

In [251]:
replacement_map = {
    pd.Timestamp('2023-09-29'): pd.Timestamp('2023-09-30'),
    pd.Timestamp('2024-03-29'): pd.Timestamp('2024-03-31'),
    pd.Timestamp('2024-06-28'): pd.Timestamp('2024-06-30')
}

df_filtered['End_month'] = df_filtered['End_month'].replace(replacement_map)

In [252]:
len(df_filtered)

1434369

In [253]:
df_filtered = df_filtered.drop_duplicates(subset=['End_month','Measures_normalized','Amount','BVU'], keep='last')

In [254]:
len(df_filtered)

1434369

In [255]:
# MAKING QUARTERS====

In [256]:
mask = df_filtered['End_month'].dt.month.isin([3, 6, 9, 12])

df_filtered['Quarter'] = pd.NaT
df_filtered.loc[mask, 'Quarter'] = df_filtered.loc[mask, 'End_month']

In [257]:
df_filtered.loc[:, 'Quarter_2'] = (
    df_filtered['Quarter']
        .dt.to_period('Q')
        .astype(str)
        .str.replace('Q', '-Q', regex=False)
)

In [258]:
# Берем только квартальные данные ---------------------------------------------------------------

In [ ]:
df_filtered

In [260]:
df_filtered = df_filtered[df_filtered['Quarter'].notna()]

In [261]:
# Пересчитывает коммулятивные цифры на квартальные для статьи расходов и доходов ---------------------------------------------------------------------------

In [262]:
df_filtered.SourceFile.unique()

<ArrowStringArray>
[                   'Assets_OLD.xlsx',                    'Assets_MID.xlsx',
                    '1_0 Активы.xlsx',                        '1_1 СП.xlsx',
                        '1_2 СП.xlsx',               'Liabilities_OLD.xlsx',
               'Liabilities_MID.xlsx',             '2_0 Обязательства.xlsx',
                  '2_1 Депозиты.xlsx',             '2_2 Обязательства.xlsx',
                   '2_4 Капитал.xlsx',       '4_0 Регуляторные данные.xlsx',
                        'IE_OLD.xlsx',                        'IE_MID.xlsx',
 '3_0 Статьи доходов и расходов.xlsx',                 '3_1 Чист дохд.xlsx',
                   '3_2 Расходы.xlsx']
Length: 17, dtype: str

In [263]:

files_to_adjust = [
    'IE_OLD.xlsx', 'IE_MID.xlsx',
    '3_0 Статьи доходов и расходов.xlsx', '3_1 Чист дохд.xlsx', '3_2 Расходы.xlsx'
]

df = df_filtered.copy()
df['End_month'] = pd.to_datetime(df['End_month'])
df['Year'] = df['End_month'].dt.year

mask_adjust = df['SourceFile'].isin(files_to_adjust)

# Function to compute quarterly increments from cumulative
def quarterly_from_cumulative(group):
    group = group.sort_values('End_month').copy()
    group['Amount_new'] = group['Amount']

    prev_cum = 0
    for idx, row in group.iterrows():
        month = row['End_month'].month
        cum_amount = row['Amount']

        if month in [3, 6, 9, 12]:  # quarter-end months
            q_amount = cum_amount - prev_cum
            group.at[idx, 'Amount_new'] = q_amount
            prev_cum = cum_amount
        else:
            group.at[idx, 'Amount_new'] = row['Amount']  # keep original

    return group

# Apply per Bank + Measure + Year
df_adj = df[mask_adjust].groupby(['BVU', 'Measures_normalized', 'Year'], group_keys=False).apply(quarterly_from_cumulative)

# Update original df
df.loc[mask_adjust, 'Amount'] = df_adj['Amount_new']


In [264]:
# now we r creating rolling 4quarters amounts

In [ ]:
# Corrections from Talgat 01/03/2026
used_measures = [
    'Активы',
    'Риск. активы и усл. обяз-ва',
    'Коэф.дост-ти СК (k2)',
    'Обяз-ва перед клиентами',
    'Текущие и карт-счета, тыс.тенге',
    'Вклады до востребования юр. и физ. лиц, тыс.тенге',
    'Всего вклады юр. и физ. лиц, тыс.тенге',
    'Активы, приносящие доход',
    'Операционные расходы (на конец периода, тыс. тенге)',
    'Чистый % доход, связанный с получением вознаграждения',
    'Доходы, не связанные с получением вознаграждения (на конец периода, тыс. тенге)',
    'Расходы, не связанные с выплатой вознаграждения (на конец периода, тыс. тенге)',
    'Расходы, связанные с выплатой вознаграждения (на конец периода, тыс. тенге)',
    'Обяз-ва, связ. с выпл.%возн.',
    'Высоколиквидные активы',
    'Итого ссудный портфель',
    'Чистый доход до уплаты подоходного налога (на конец периода, тыс. тенге)',
    'Чистый доход после уплаты подоходного налога (на конец периода, тыс. тенге)',
    'Собственный капитал',
    'Ценные бумаги (на конец периода, тыс. тенге)',
    'Производные финансовые инструменты (на конец периода, тыс. тенге)',
    'Сумма займов, по которым просроченная задолженность свыше 90 дней',
    'Расходы на формирование провизий (на конец периода, тыс. тенге)',
    'Расходы по выплате подоходного налога (на конец периода, тыс. тенге)',
    'Обяз-ва БВУ в ин.валюте',
    'Обязательства',
    'Требования к НБ (на конец периода, тыс. тенге)',
    'Корр. Счета и вклады, размещенные в других БВУ и НБО (на конец периода, тыс. тенге)',
    'Деньги и драг. металлы (на конец периода, тыс. тенге)'
]
df_used = df[df['Measures_normalized'].isin(used_measures)]
df_used = df_used[(df_used.duplicated(subset=['Date', 'BVU', 'Measures_normalized'], keep=False))]
df_used.BVU.unique()

In [266]:
# First correct the HLA thing with all 7 banks
target_measure = 'Высоколиквидные активы'
preferred_source = '4_0 Регуляторные данные.xlsx'

df_target = df[df['Measures_normalized'] == target_measure].copy()
df_other = df[df['Measures_normalized'] != target_measure].copy()

df_target = df_target.sort_values(
    by='SourceFile',
    key=lambda x: x != preferred_source)

df_target_clean = df_target.drop_duplicates(
    subset=['Date', 'BVU', 'Measures_normalized'],
    keep='first')

df_clean = pd.concat([df_other, df_target_clean], ignore_index=True)

In [ ]:
df_used = df_clean[df_clean['Measures_normalized'].isin(used_measures)]
df_used = df_used[(df_used.duplicated(subset=['Date', 'BVU', 'Measures_normalized'], keep=False))]
df_used.BVU.unique()

In [269]:
df_used = df_clean[df_clean['Measures_normalized'].isin(used_measures)]
df_used = df_used[(df_used.duplicated(subset=['Date', 'BVU', 'Measures_normalized'], keep=False))]
df_used.BVU.unique()

<ArrowStringArray>
[]
Length: 0, dtype: str

In [270]:
df = df_clean.copy()

In [271]:
def rolling_4q_per_bvu_measure_safe(df, bvu_col='BVU', measure_col='Measures_normalized', value_col='Amount', date_col='End_month'):
    """
    Calculate 4-quarter rolling sum per BVU and Measure safely.
    Works even if index is non-default or mixed.
    """
    df = df.sort_values([bvu_col, measure_col, date_col])
    
    # Group by BVU + Measure
    grouped = df.groupby([bvu_col, measure_col], group_keys=False)
    
    # Apply rolling sum safely using .transform
    df['Value_4Q_sum'] = grouped[value_col].transform(lambda x: x.rolling(window=4, min_periods=1).sum())
    
    return df


df = rolling_4q_per_bvu_measure_safe(df)

In [272]:
# now calculating rolling window of 5 quarters --------------------------------------------

In [273]:
def rolling_5q_avg_per_bvu_measure(df, bvu_col='BVU', measure_col='Measures_normalized', value_col='Amount', date_col='End_month'):
    """
    Calculate 5-quarter rolling average per BVU and Measure, including current quarter.
    """
    # Sort by BVU, Measure, and Date
    df = df.sort_values([bvu_col, measure_col, date_col])
    
    # Group by BVU + Measure and apply rolling mean
    df[f'{value_col}_5Q_avg'] = (
        df.groupby([bvu_col, measure_col])[value_col]
          .transform(lambda x: x.rolling(window=5, min_periods=1).mean())
    )
    
    return df


df = rolling_5q_avg_per_bvu_measure(df)


In [274]:
# now calculating Q/Q growth --------------------------------------------

In [275]:
def q_to_q_growth(df, bvu_col='BVU', measure_col='Measures_normalized', value_col='Amount', date_col='End_month'):
    # Sort by BVU, Measure, and Date
    df = df.sort_values([bvu_col, measure_col, date_col])
    
    # Group by BVU + Measure and calculate pct_change
    df[f'{value_col}_QoQ_growth'] = (
        df.groupby([bvu_col, measure_col])[value_col]
          .transform(lambda x: x.pct_change())  # default: (current - prev)/prev
    )
    
    return df


df = q_to_q_growth(df)

In [276]:
# saving file ------------------------------------

In [ ]:
# df.to_parquet("cleaned_df.parquet", index=False)
df.to_parquet(".parquet", index=False) # Talgat 28.02.2026